# RigTech · Inferência em CearaMirim.tif (ciclo 1)

**Objetivo**: rodar o modelo já treinado no ciclo 1 (`train_ckpt_c1.pt`) sobre uma imagem **de teste** (`CearaMirim.tif`) — nunca vista pelo modelo — e gerar polígonos de folha_larga / folha_estreita previstos.

**Nada é retreinado**: o backbone DINOv3 é congelado por design, e a head Transformer é carregada com os pesos salvos no ciclo 1 (`best_state` = melhor época de validação em macro-F1). Só se muda a imagem de entrada.

**Pré-requisitos no Drive**:
- `MyDrive/CearaMirim.tif` — ortomosaico RGB (bandas 1,2,3).
- `MyDrive/rigtech_ciclos/train_ckpt_c1.pt` — checkpoint gerado pelo `runner_colab_continuacao`.
- Secret `HF_TOKEN` no Colab (chave lateral) — `dinov3-vitl16-pretrain-sat493m` é *gated*.

Runtime → GPU (T4 basta pra inferência).

## 1. Clonar repo + instalar dependências

Traz o `src/infer.py` mais recente. Se der conflito de versão, **Runtime → Restart runtime** antes da próxima célula.

In [ ]:
!git clone https://github.com/Rigtech-Solutions/rigtech-weed-cycle.git /content/rigtech || (cd /content/rigtech && git pull)

!pip -q uninstall -y pandas numpy
!pip -q install 'numpy>=1.26,<2.2' 'pandas>=2.2,<2.3'
!pip -q install rasterio geopandas shapely scikit-image tqdm transformers torch huggingface_hub

# IMPORTANTE: se aparecer erro de versao ao importar geopandas/pandas,
# faca Runtime -> Restart runtime (Ctrl+M .) antes de rodar as proximas.

## 2. Login no Hugging Face + montar Drive

In [ ]:
import os
from huggingface_hub import login

_hf_token = None
try:
    from google.colab import userdata
    _hf_token = userdata.get('HF_TOKEN')
except Exception:
    _hf_token = os.environ.get('HF_TOKEN')

if _hf_token:
    login(token=_hf_token)
    print('Login HF OK.')
else:
    login()

from google.colab import drive
drive.mount('/content/drive')

## 3. Configuração da rodada

Só variáveis — nada muda no modelo.

In [ ]:
CICLO = 1  # qual checkpoint de treino usar

CKPT = f'/content/drive/MyDrive/rigtech_ciclos/train_ckpt_c{CICLO}.pt'
TIF  = '/content/drive/MyDrive/CearaMirim.tif'
OUT_PREFIX = f'/content/drive/MyDrive/rigtech_ciclos/pred_cearamirim_c{CICLO}'

# sliding window
STRIDE = 112              # 50% de overlap (CROP=224)
WEED_THRESHOLD = 0.6      # prob min de folha_larga OU folha_estreita pra tile virar positivo
MERGE_BUFFER = 1.0        # unidades do CRS do raster (UTM: metros) -- fecha gaps entre tiles
MIN_POLY_AREA = 5.0       # descarta poligonos com area < isso (unidades^2 do CRS)
BATCH = 32

assert os.path.exists(CKPT), f'checkpoint nao encontrado: {CKPT}'
assert os.path.exists(TIF), f'imagem nao encontrada: {TIF}'
print('checkpoint:', CKPT)
print('imagem    :', TIF)
print('saidas    :', OUT_PREFIX + '.{csv,geojson,_polygons.geojson}')

## 4. Rodar inferência

Sliding window sobre o raster inteiro. Se depois quiser restringir só à plantação, gera um geojson e passa `--mask <caminho>`.

In [ ]:
%cd /content/rigtech
!python -m src.infer \
    --ckpt {CKPT} \
    --tif  {TIF} \
    --sliding --stride {STRIDE} \
    --weed-threshold {WEED_THRESHOLD} \
    --merge-buffer {MERGE_BUFFER} \
    --min-polygon-area {MIN_POLY_AREA} \
    --batch {BATCH} \
    --out-prefix {OUT_PREFIX}

## 5. Resumo dos resultados

Conta previsões por classe no CSV e polígonos agregados no geojson.

In [ ]:
import pandas as pd
import geopandas as gpd

df = pd.read_csv(OUT_PREFIX + '.csv')
print(f'total de tiles avaliados: {len(df)}')
print('\ndistribuicao por classe predita:')
print(df['pred_class'].value_counts())

print('\nprob media por classe:')
print(df[['prob_cultivo', 'prob_folha_larga', 'prob_folha_estreita']].mean())

polys = gpd.read_file(OUT_PREFIX + '_polygons.geojson')
print(f'\npoligonos agregados: {len(polys)}')
if len(polys):
    print(polys.groupby('class').agg(n=('class', 'size'), area_total=('area_crs', 'sum')))

## 6. Preview visual (opcional)

Plota os polígonos previstos sobre um thumbnail do raster pra inspeção rápida no notebook. Pra análise fina, abre o `_polygons.geojson` no QGIS por cima do `.tif`.

In [ ]:
import matplotlib.pyplot as plt
import rasterio
from rasterio.enums import Resampling
import geopandas as gpd

polys = gpd.read_file(OUT_PREFIX + '_polygons.geojson')

with rasterio.open(TIF) as src:
    scale = max(1, max(src.width, src.height) // 2000)
    out_shape = (3, src.height // scale, src.width // scale)
    thumb = src.read([1, 2, 3], out_shape=out_shape, resampling=Resampling.average)
    left, bottom, right, top = src.bounds
    raster_crs = src.crs

thumb = thumb.transpose(1, 2, 0)
if thumb.dtype != 'uint8':
    thumb = (255 * (thumb / max(thumb.max(), 1))).clip(0, 255).astype('uint8')

polys_native = polys.to_crs(raster_crs) if raster_crs is not None else polys

fig, ax = plt.subplots(figsize=(12, 12))
ax.imshow(thumb, extent=(left, right, bottom, top))
colors = {'folha_larga': 'red', 'folha_estreita': 'yellow'}
for cls, color in colors.items():
    sub = polys_native[polys_native['class'] == cls]
    if len(sub):
        sub.boundary.plot(ax=ax, color=color, linewidth=1.2, label=f'{cls} ({len(sub)})')
ax.set_title(f'CearaMirim — previsoes ciclo {CICLO}')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()